# M5 Data Cleaning & Preparation

**What this notebook does:**
1. Parse types and fill missing values  
2. Downcast columns to save memory (~440 MB saved on sales alone)  
3. Detect leading zeros (items not yet on shelf)  
4. Melt sales from wide → long and merge all three tables  
5. Forward-fill price gaps from the merge  
6. Placeholder for joining external datasets

In [3]:
import os, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

## 1. Configuration

In [2]:
# Update DATA_DIR to point to your local copy of the Kaggle M5 data.
# Default: place the CSV files in  <repo_root>/data/raw/
DATA_DIR = Path("../data/raw")

CALENDAR_PATH = DATA_DIR / "calendar.csv"
PRICES_PATH   = DATA_DIR / "sell_prices.csv"

if (DATA_DIR / "sales_train_evaluation.csv").exists():
    SALES_PATH = DATA_DIR / "sales_train_evaluation.csv"
else:
    SALES_PATH = DATA_DIR / "sales_train_validation.csv"

FAST_MODE    = False    # Sample N_SERIES for quick runs; set False for full data
N_SERIES     = 2000
HISTORY_DAYS = 500     # Days of history to keep in long format
VALID_DAYS   = 28
RANDOM_STATE = 1234

OUTPUT_DIR = Path("./cleaned_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Sales file:", SALES_PATH.name)
print("Fast mode:", FAST_MODE)


Sales file: sales_train_evaluation.csv
Fast mode: False


## 2. Load Raw Data

In [4]:
calendar_raw = pd.read_csv(CALENDAR_PATH)
prices_raw   = pd.read_csv(PRICES_PATH)
sales_raw    = pd.read_csv(SALES_PATH)

print("calendar:", calendar_raw.shape)
print("prices:  ", prices_raw.shape)
print("sales:   ", sales_raw.shape)

calendar: (1969, 14)
prices:   (6841121, 4)
sales:    (30490, 1947)


## 3. Clean Calendar

- Parse `date` to datetime  
- Fill event NaNs with `"NoEvent"` (1,807 of 1,969 days have no event)  
- Downcast integer columns

In [5]:
calendar = calendar_raw.copy()

# Parse date
calendar["date"] = pd.to_datetime(calendar["date"])

# Fill event NaNs — these are not missing data, they are "no event happened"
for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    print(f"  {col}: filled {calendar[col].isna().sum()} NaN → 'NoEvent'")
    calendar[col] = calendar[col].fillna("NoEvent")

# Downcast
for col in ["wday", "month", "snap_CA", "snap_TX", "snap_WI"]:
    calendar[col] = calendar[col].astype(np.int8)
calendar["year"] = calendar["year"].astype(np.int16)

# Add integer day index for efficient operations downstream
calendar["d_num"] = calendar["d"].str.extract(r"(\d+)").astype(np.int16)

print(f"\n✓ Calendar cleaned: {calendar.shape}")

  event_name_1: filled 1807 NaN → 'NoEvent'
  event_type_1: filled 1807 NaN → 'NoEvent'
  event_name_2: filled 1964 NaN → 'NoEvent'
  event_type_2: filled 1964 NaN → 'NoEvent'

✓ Calendar cleaned: (1969, 15)


## 4. Clean Prices

- Downcast `sell_price` from float64 → float32  
- Quick sanity check

In [6]:
prices = prices_raw.copy()
prices["sell_price"] = prices["sell_price"].astype(np.float32)

print(f"Price range: ${prices['sell_price'].min():.2f} – ${prices['sell_price'].max():.2f}")
print(f"NaN prices: {prices['sell_price'].isna().sum()}")
print(f"✓ Prices cleaned: {prices.shape}")

Price range: $0.01 – $107.32
NaN prices: 0
✓ Prices cleaned: (6841121, 4)


## 5. Clean Sales

- Downcast 1,941 day columns from int64 → int16 (saves ~440 MB)  
- Compute `first_sale_day` per series to identify leading zeros  
  - 77% of series have leading zeros — these are items that weren't on the shelf yet  
  - Downstream notebooks can use `is_active` to exclude pre-shelf zeros from training

In [7]:
sales = sales_raw.copy()
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
d_cols = sorted([c for c in sales.columns if c.startswith("d_")],
                key=lambda x: int(x.split("_")[1]))

# Downcast day columns
mem_before = sales[d_cols].memory_usage(deep=True).sum() / 1e6
for col in d_cols:
    sales[col] = sales[col].astype(np.int16)
mem_after = sales[d_cols].memory_usage(deep=True).sum() / 1e6
print(f"Memory: {mem_before:.0f} MB → {mem_after:.0f} MB (saved {mem_before - mem_after:.0f} MB)")

# Compute first non-zero sale day per series
mat = sales[d_cols].values
first_nonzero = np.argmax(mat > 0, axis=1)
all_zero = mat.sum(axis=1) == 0
first_nonzero[all_zero] = mat.shape[1]

series_meta = sales[id_cols].copy()
series_meta["first_sale_day"] = first_nonzero + 1  # 1-indexed to match d_1, d_2, ...
series_meta["has_leading_zeros"] = (first_nonzero > 0).astype(np.int8)

print(f"Series with leading zeros: {series_meta['has_leading_zeros'].sum()} / {len(series_meta)}")
print(f"✓ Sales cleaned: {sales.shape}")

Memory: 473 MB → 118 MB (saved 355 MB)
Series with leading zeros: 23511 / 30490
✓ Sales cleaned: (30490, 1947)


In [8]:
# Optional: sample for fast iteration
if FAST_MODE:
    sampled_ids = sales["id"].sample(n=min(N_SERIES, len(sales)), random_state=RANDOM_STATE).values
    sales = sales[sales["id"].isin(sampled_ids)].reset_index(drop=True)
    series_meta = series_meta[series_meta["id"].isin(sampled_ids)].reset_index(drop=True)
    print(f"FAST_MODE: sampled {len(sales)} series")
else:
    print(f"Full data: {len(sales)} series")

Full data: 30490 series


## 6. Melt to Long Format & Merge

- Keep only the most recent `HISTORY_DAYS + VALID_DAYS` columns  
- Melt wide → long  
- Left-join calendar and prices  
- Forward/back-fill NaN prices (items that appear mid-history have no price for early days)  
- Create `is_active` flag and state-specific `snap_flag`

In [9]:
# Keep recent days only
keep_d = d_cols[-(HISTORY_DAYS + VALID_DAYS):]
print(f"Keeping {len(keep_d)} day columns: {keep_d[0]} → {keep_d[-1]}")

# Melt
long_df = sales[id_cols + keep_d].melt(
    id_vars=id_cols, value_vars=keep_d, var_name="d", value_name="demand"
)
long_df["demand"] = long_df["demand"].astype(np.int16)

# Merge calendar
cal_merge = ["d", "d_num", "date", "wm_yr_wk", "wday", "month", "year", "weekday",
             "snap_CA", "snap_TX", "snap_WI",
             "event_name_1", "event_type_1", "event_name_2", "event_type_2"]
long_df = long_df.merge(calendar[cal_merge], on="d", how="left")
long_df["date"] = pd.to_datetime(long_df["date"])

# Merge prices
long_df = long_df.merge(
    prices[["store_id", "item_id", "wm_yr_wk", "sell_price"]],
    on=["store_id", "item_id", "wm_yr_wk"], how="left"
)

# Sort by series + time
long_df = long_df.sort_values(["id", "d_num"]).reset_index(drop=True)

print(f"Long format: {long_df.shape}")
print(f"Price NaN before fill: {long_df['sell_price'].isna().sum():,}")

Keeping 528 day columns: d_1414 → d_1941
Long format: (16098720, 23)
Price NaN before fill: 119,168


In [10]:
# Forward/back-fill price gaps within each series
long_df["sell_price"] = long_df.groupby("id")["sell_price"].transform(
    lambda s: s.ffill().bfill()
)
long_df["sell_price"] = long_df["sell_price"].fillna(0).astype(np.float32)
print(f"Price NaN after fill: {long_df['sell_price'].isna().sum()}")

# is_active: whether the item was on shelf at this date
first_sale = series_meta.set_index("id")["first_sale_day"]
long_df["is_active"] = (long_df["d_num"] >= long_df["id"].map(first_sale)).astype(np.int8)
print(f"Active rows: {long_df['is_active'].mean()*100:.1f}%")

# snap_flag: state-specific SNAP indicator
long_df["snap_flag"] = np.int8(0)
for state, col in [("CA", "snap_CA"), ("TX", "snap_TX"), ("WI", "snap_WI")]:
    mask = long_df["state_id"] == state
    long_df.loc[mask, "snap_flag"] = long_df.loc[mask, col].astype(np.int8)

print(f"\n✓ Long-format table ready: {long_df.shape}")
display(long_df.head())

Price NaN after fill: 0
Active rows: 99.2%

✓ Long-format table ready: (16098720, 25)


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,d_num,date,wm_yr_wk,wday,month,year,weekday,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,is_active,snap_flag
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1414,1,1414,2014-12-12,11445,7,12,2014,Friday,0,1,1,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1415,2,1415,2014-12-13,11446,1,12,2014,Saturday,0,1,0,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1416,2,1416,2014-12-14,11446,2,12,2014,Sunday,0,0,1,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1417,1,1417,2014-12-15,11446,3,12,2014,Monday,0,1,1,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1418,1,1418,2014-12-16,11446,4,12,2014,Tuesday,0,0,0,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0


## 7. External Dataset Joins (Placeholder)

Uncomment any block below to join external data.  
Supply the matching CSV file in your `DATA_DIR`.

In [ ]:
# ==============================================================================
# EXTERNAL DATASETS — UNCOMMENT AND ADAPT AS NEEDED
# ==============================================================================

# ---------- Unemployment by state (monthly, from BLS) ----------
# unemployment = pd.read_csv(DATA_DIR / "state_unemployment.csv")
# unemployment["date"] = pd.to_datetime(unemployment["date"])
# unemployment["ym"] = unemployment["date"].dt.to_period("M")
# long_df["ym"] = long_df["date"].dt.to_period("M")
# long_df = long_df.merge(unemployment[["ym", "state_id", "unemployment_rate"]],
#                         on=["ym", "state_id"], how="left")
# long_df.drop(columns=["ym"], inplace=True)


# ---------- Daily weather by state (from NOAA / Visual Crossing) ----------
# weather = pd.read_csv(DATA_DIR / "daily_weather_by_state.csv")
# weather["date"] = pd.to_datetime(weather["date"])
# long_df = long_df.merge(weather[["date", "state_id", "temp_max", "precipitation"]],
#                         on=["date", "state_id"], how="left")


# ---------- CPI / inflation (monthly, from BLS) ----------
# cpi = pd.read_csv(DATA_DIR / "us_cpi_monthly.csv")
# cpi["date"] = pd.to_datetime(cpi["date"])
# cpi["ym"] = cpi["date"].dt.to_period("M")
# long_df["ym"] = long_df["date"].dt.to_period("M")
# long_df = long_df.merge(cpi[["ym", "cpi_index"]], on="ym", how="left")
# long_df["real_price"] = long_df["sell_price"] / long_df["cpi_index"] * 100
# long_df.drop(columns=["ym"], inplace=True)


# ---------- Google Trends by product category ----------
# trends = pd.read_csv(DATA_DIR / "google_trends_by_category.csv")
# trends["date"] = pd.to_datetime(trends["date"])
# long_df = long_df.merge(trends[["date", "cat_id", "search_interest"]],
#                         on=["date", "cat_id"], how="left")


# ---------- State-level holidays (via `pip install holidays`) ----------
# import holidays
# for st in ["CA", "TX", "WI"]:
#     hols = holidays.US(years=range(2011, 2017), state=st)
#     calendar[f"holiday_{st}"] = calendar["date"].dt.date.isin(hols).astype(np.int8)
# # Then re-merge the new columns into long_df


print("No external datasets joined (placeholder only).")

## 8. Export

Outputs:
- `long_df_clean.parquet` — cleaned long-format table  
- `series_meta.parquet` — per-series metadata (first_sale_day, leading zeros)  

Downstream notebooks load these instead of re-running the raw pipeline.

In [ ]:
# Convert category types to string for Parquet compatibility
for col in ["item_id", "dept_id", "cat_id", "store_id", "state_id", "weekday",
            "event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    if col in long_df.columns:
        long_df[col] = long_df[col].astype(str)
    if col in series_meta.columns:
        series_meta[col] = series_meta[col].astype(str)

# Export as Parquet (fast + compressed). Falls back to CSV if pyarrow is missing.
try:
    long_df.to_parquet(OUTPUT_DIR / "long_df_clean.parquet", index=False)
    series_meta.to_parquet(OUTPUT_DIR / "series_meta.parquet", index=False)
    fmt = "parquet"
except ImportError:
    print("pyarrow not found — falling back to CSV (install pyarrow for faster I/O)")
    long_df.to_csv(OUTPUT_DIR / "long_df_clean.csv", index=False)
    series_meta.to_csv(OUTPUT_DIR / "series_meta.csv", index=False)
    fmt = "csv"

for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

print(f"\n--- Summary ---")
print(f"Format: {fmt}")
print(f"Series: {long_df['id'].nunique():,}")
print(f"Rows: {len(long_df):,}")
print(f"Date range: {long_df['date'].min().date()} → {long_df['date'].max().date()}")
print(f"Active rows: {long_df['is_active'].mean()*100:.1f}%")
print(f"Zero demand: {(long_df['demand']==0).mean()*100:.1f}%")

## How to Load in Modeling Notebooks

```python
from pathlib import Path
import pandas as pd

CLEAN_DIR = Path("./cleaned_data")

# Parquet (if pyarrow is installed)
long_df     = pd.read_parquet(CLEAN_DIR / "long_df_clean.parquet")
series_meta = pd.read_parquet(CLEAN_DIR / "series_meta.parquet")

# Or CSV fallback
# long_df     = pd.read_csv(CLEAN_DIR / "long_df_clean.csv", parse_dates=["date"])
# series_meta = pd.read_csv(CLEAN_DIR / "series_meta.csv")

# Optional: exclude pre-shelf zeros
long_df = long_df[long_df["is_active"] == 1].reset_index(drop=True)
```